<a href="https://colab.research.google.com/github/MihGok/DeepLearning/blob/main/CodeBert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 21.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.9 MB/s eta 0:00:00


In [ ]:
! git clone -q https://github.com/microsoft/CodeXGLUE.git

In [ ]:
from datasets import  DatasetDict
from datasets import load_dataset
train_dataset = load_dataset("code_x_glue_ct_code_to_text", "ruby", split="train[:2500]")
test_dataset = load_dataset("code_x_glue_ct_code_to_text", "ruby", split="test[:700]")
validation_dataset = load_dataset("code_x_glue_ct_code_to_text", "ruby", split="validation[:200]")
# Создание объекта DatasetDict и добавление загруженных наборов данных
dataset = DatasetDict({
    'train': train_dataset,
    'validation': validation_dataset,
    'test': test_dataset
})

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/24927 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1400 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1261 [00:00<?, ? examples/s]

In [ ]:
import json
# Преобразуем данные в формат JSONL и сохраняем их в файлы
def save_data_to_jsonl(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for item in data:
            json.dump(item, f)
            f.write('\n')

# Создаем списки для тренировочных, тестовых и валидационных данных
train_data = dataset['train']
test_data = dataset['test']
validation_data = dataset['validation']

# Сохраняем данные в соответствующие файлы
save_data_to_jsonl(train_data, 'train.jsonl')
save_data_to_jsonl(test_data, 'test.jsonl')
save_data_to_jsonl(validation_data, 'validation.jsonl')

In [ ]:
import pandas as pd

In [ ]:
from dataclasses import dataclass
@dataclass
class ConfigurationModel:
	learning_rate : float
	batch_size : int
	beam : int
	test_file : str
	source_size : int
	target_size : int
	path_to_data_directory : str
	path_to_output_data_directory : str
	train_file : str
	dev_file : str
	count_epochs : int
	pretrained_model : str


configuration_codetext_model = ConfigurationModel(
	learning_rate = 5e-5,
	batch_size = 8,
	beam = 10,
	source_size = 256,
	target_size = 512,
	path_to_data_directory = '.',
	path_to_output_data_directory = 'model_for_java',
	train_file = '/content/train.jsonl',
	dev_file = '/content/validation.jsonl',
	test_file = '/content/test.jsonl',
	count_epochs = 6,
	pretrained_model = 'microsoft/codebert-base',
)
configuration_codetext_model

ConfigurationModel(learning_rate=5e-05, batch_size=8, beam=10, test_file='/content/test.jsonl', source_size=256, target_size=512, path_to_data_directory='.', path_to_output_data_directory='model_for_java', train_file='/content/train.jsonl', dev_file='/content/validation.jsonl', count_epochs=6, pretrained_model='microsoft/codebert-base')

In [ ]:
!python /content/CodeXGLUE/Code-Text/code-to-text/code/run.py \
	--do_train \
	--do_eval \
	--do_lower_case \
	--model_type roberta \
	--model_name_or_path {configuration_codetext_model.pretrained_model} \
	--train_filename {configuration_codetext_model.train_file} \
	--dev_filename {configuration_codetext_model.dev_file} \
	--output_dir {configuration_codetext_model.path_to_output_data_directory} \
	--max_source_length {configuration_codetext_model.source_size} \
	--max_target_length {configuration_codetext_model.target_size} \
	--beam_size {configuration_codetext_model.beam} \
	--train_batch_size {configuration_codetext_model.batch_size} \
	--eval_batch_size {configuration_codetext_model.batch_size} \
	--learning_rate {configuration_codetext_model.learning_rate} \
	--num_train_epochs {configuration_codetext_model.count_epochs}

03/17/2024 11:37:51 - INFO - __main__ -   Namespace(model_type='roberta', model_name_or_path='microsoft/codebert-base', output_dir='model_for_java', load_model_path=None, train_filename='/content/train.jsonl', dev_filename='/content/validation.jsonl', test_filename=None, config_name='', tokenizer_name='', max_source_length=256, max_target_length=512, do_train=True, do_eval=True, do_test=False, do_lower_case=True, no_cuda=False, train_batch_size=8, eval_batch_size=8, gradient_accumulation_steps=1, learning_rate=5e-05, beam_size=10, weight_decay=0.0, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=6, max_steps=-1, eval_steps=-1, train_steps=-1, warmup_steps=0, local_rank=-1, seed=42)
03/17/2024 11:37:51 - WARNING - __main__ -   Process rank: -1, device: cuda, n_gpu: 1, distributed training: False
config.json: 100% 498/498 [00:00<00:00, 2.13MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 146kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 24.0MB/s]
merges.txt: 100% 456k/45

In [ ]:
binary_model_file = '/content/model_for_java/checkpoint-best-bleu/pytorch_model.bin'
!python  /content/CodeXGLUE/Code-Text/code-to-text/code/run.py \
    --do_test \
    --model_type roberta \
    --model_name_or_path microsoft/codebert-base \
    --load_model_path {binary_model_file} \
    --dev_filename {configuration_codetext_model.dev_file} \
    --test_filename {configuration_codetext_model.test_file} \
    --output_dir {configuration_codetext_model.path_to_output_data_directory} \
    --max_source_length {configuration_codetext_model.source_size} \
    --max_target_length {configuration_codetext_model.target_size} \
    --beam_size {configuration_codetext_model.beam} \
    --eval_batch_size {configuration_codetext_model.batch_size}

03/17/2024 12:17:20 - INFO - __main__ -   Namespace(model_type='roberta', model_name_or_path='microsoft/codebert-base', output_dir='model_for_java', load_model_path='/content/model_for_java/checkpoint-best-bleu/pytorch_model.bin', train_filename=None, dev_filename='/content/validation.jsonl', test_filename='/content/test.jsonl', config_name='', tokenizer_name='', max_source_length=256, max_target_length=512, do_train=False, do_eval=False, do_test=True, do_lower_case=False, no_cuda=False, train_batch_size=8, eval_batch_size=8, gradient_accumulation_steps=1, learning_rate=5e-05, beam_size=10, weight_decay=0.0, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=3, max_steps=-1, eval_steps=-1, train_steps=-1, warmup_steps=0, local_rank=-1, seed=42)
03/17/2024 12:17:20 - WARNING - __main__ -   Process rank: -1, device: cuda, n_gpu: 1, distributed training: False
03/17/2024 12:17:23 - INFO - __main__ -   reload model from /content/model_for_java/checkpoint-best-bleu/pytorch_model.bin
03

In [ ]:
path_to_gold = '/content/model_for_java/dev.gold'
path_to_output = '/content/model_for_java/dev.output'

In [ ]:
def read_result_txt_file(txt_file: str)-> list:
	with open(txt_file) as file:		return [' '.join(line.rstrip().replace('\t', ' ').split(' ')[1:]) for line in file]

In [ ]:
#true comments and predicted
true_sent = read_result_txt_file(path_to_gold)
pred_sent = read_result_txt_file(path_to_output)
result_data_frame = pd.DataFrame(
    {
      'true' : true_sent,
      'pred' : pred_sent
    }
  )

for i in range(8):
  print('Correct:', result_data_frame['true'][i])
  print('Predicted', result_data_frame['pred'][i],'\n')
result_data_frame.head(10)

Correct: Handles resources such as tickets . Any options are passed to the underlying collection except reload which disregards memoization and creates a new Collection instance .
Predicted Create a new cache . 

Correct: Validates Time argument .
Predicted Convert value to a hash 

Correct: Hash of cookies extracted from response headers .
Predicted Return a cookie request to the cookie object . 

Correct: A helper method to dispatch to an ItemId OccurrenceItemId or a RecurringMasterItemId
Predicted Determine if there is a single id 

Correct: Check whether or not to log payloads based on log level .
Predicted Logs a message message 

Correct: Executes the SOAP request with original SOAP name .
Predicted Perform API request 

Correct: Convert AdManagerDateTime into a hash representation which can be consumed by the Ad Manager API . E . g . a hash that can be passed as PQL DateTime variables .
Predicted Create a new object . 

Correct: Auxiliary method to recurse through a hash and con

,true,pred
0,Handles resources such as tickets . Any option...,Create a new cache .
1,Validates Time argument .,Convert value to a hash
2,Hash of cookies extracted from response headers .,Return a cookie request to the cookie object .
3,A helper method to dispatch to an ItemId Occur...,Determine if there is a single id
4,Check whether or not to log payloads based on ...,Logs a message message
5,Executes the SOAP request with original SOAP n...,Perform API request
6,Convert AdManagerDateTime into a hash represen...,Create a new object .
7,Auxiliary method to recurse through a hash and...,Converts a hash keys to hash
8,Build the AttachmentIds element,Returns an array of attributes
9,Validates Array argument .,Check if a given args
